<a href="https://colab.research.google.com/github/harsh3-web/Ecommerce-Recommender/blob/main/notebooks/04_evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 04 · Evaluation

Leave-last-purchase-out split (each user's most recent purchase is hidden, and its browse removed to prevent leakage), users split into validation (to tune alpha) and test. Compares Random, Popular, the original methods, item-CF, content and the hybrid with **HR@10** and **NDCG@10**, plus a diversity check. Final model: hybrid with alpha = 0.75.

In [ ]:
!git clone https://github.com/harsh3-web/Ecommerce-Recommender.git
%cd Ecommerce-Recommender
# torch_geometric is needed because we also test the ORIGINAL functions from legacy/model.py
!pip install -q sentence-transformers torch_geometric

Cloning into 'Ecommerce-Recommender'...
remote: Enumerating objects: 23, done.
remote: Counting objects: 100% (23/23), done.
remote: Compressing objects: 100% (23/23), done.
remote: Total 23 (delta 8), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (23/23), 49.71 KiB | 4.97 MiB/s, done.
Resolving deltas: 100% (8/8), done.
/content/Ecommerce-Recommender
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 24.3 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!cp /content/drive/MyDrive/Ecommerce-Recommender-data/*.csv .
!ls *.csv

Mounted at /content/drive
browsing_history_expanded.csv  products_expanded.csv
ground_truth_tastes.csv        purchases_expanded.csv
product_images_expanded.csv    users_expanded.csv


In [ ]:
import numpy as np
import pandas as pd

products  = pd.read_csv("products_expanded.csv")
users     = pd.read_csv("users_expanded.csv")
# parse_dates -> timestamps become real dates, so we can find each user's LAST purchase
purchases = pd.read_csv("purchases_expanded.csv", parse_dates=["timestamp"])
browsing  = pd.read_csv("browsing_history_expanded.csv", parse_dates=["timestamp"])
n_users, n_products = len(users), len(products)

In [ ]:
# sort by time, then take the LAST row per user = their most recent purchase
last = purchases.sort_values("timestamp").groupby("user_id").tail(1)

n_buys = purchases.groupby("user_id").size()          # purchases per user
eligible = n_buys[n_buys >= 2].index                  # need >= 1 purchase LEFT for training
test_items = last[last["user_id"].isin(eligible)].set_index("user_id")["product_id"]
print(len(test_items), "users will be evaluated")     # test_items[u] = the hidden product for user u

455 users will be evaluated


In [ ]:
def drop_test(df):
    """Remove every row where a user interacts with THEIR OWN hidden test product."""
    hidden = df["user_id"].map(test_items)     # hidden product for each row's user (NaN if not evaluated)
    return df[df["product_id"] != hidden].copy()

train_purchases = drop_test(purchases)
train_browsing  = drop_test(browsing)
print(f"Purchases: {len(purchases)} -> {len(train_purchases)} | Browsing: {len(browsing)} -> {len(train_browsing)}")

Purchases: 1901 -> 1446 | Browsing: 6269 -> 5814


In [ ]:
rng = np.random.default_rng(42)
eval_users = test_items.index.to_numpy()
rng.shuffle(eval_users)                       # random order (fixed by the seed)
half = len(eval_users) // 2
val_users, test_users = eval_users[:half], eval_users[half:]
print(len(val_users), "validation users |", len(test_users), "test users")

227 validation users | 228 test users


In [ ]:
def hit_at_k(ranked_ids, true_id, k=10):
    """1 if the hidden product appears anywhere in the top k, else 0."""
    return float(true_id in ranked_ids[:k])

def ndcg_at_k(ranked_ids, true_id, k=10):
    """Like hit rate, but rewards ranking the hidden product HIGHER."""
    top = list(ranked_ids[:k])
    if true_id not in top:
        return 0.0
    rank = top.index(true_id) + 1             # position 1, 2, ... 10
    return 1 / np.log2(rank + 1)              # rank 1 -> 1.0, rank 2 -> 0.63, rank 10 -> 0.29

In [ ]:
def evaluate(recommend_fn, user_ids, k=10):
    """Average HR@k and NDCG@k over users. recommend_fn(u) must return a table sorted best-first."""
    hits, ndcgs = [], []
    for u in user_ids:
        ranked = recommend_fn(u)["product_id"].tolist()
        hits.append(hit_at_k(ranked, test_items[u], k))
        ndcgs.append(ndcg_at_k(ranked, test_items[u], k))
    return {"HR@10": np.mean(hits), "NDCG@10": np.mean(ndcgs)}

In [ ]:
from scipy.sparse import csr_matrix
BROWSE_W, BUY_W = 1.0, 4.0
b = train_browsing[["user_id", "product_id"]].assign(weight=BROWSE_W)
p = train_purchases[["user_id", "product_id"]].assign(weight=BUY_W)
inter = pd.concat([b, p]).groupby(["user_id", "product_id"], as_index=False)["weight"].sum()

R = csr_matrix((inter["weight"].values, (inter["user_id"].values - 1, inter["product_id"].values - 1)),
               shape=(n_users, n_products))

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
item_sim = cosine_similarity(R.T)        # 200 x 200, built from TRAINING data only
np.fill_diagonal(item_sim, 0)

# Dictionary: user_id -> set of products they already touched (in training). Fast to look up.
seen = inter.groupby("user_id")["product_id"].apply(set).to_dict()

In [ ]:
def to_recs(scores, uid, source):
    """Array of 200 scores (product_id order) -> ranked table without already-seen products."""
    recs = products.copy()
    recs["score"], recs["source"] = scores, source
    recs = recs[~recs["product_id"].isin(seen.get(uid, set()))]   # .get -> empty set if user unknown
    return recs.sort_values("score", ascending=False)

In [ ]:
def random_recs(uid):
    return to_recs(rng.random(n_products), uid, "Random")       # random scores -> random ranking

# How often each product was bought in TRAINING; reindex fills never-bought products with 0
pop = train_purchases["product_id"].value_counts().reindex(products["product_id"], fill_value=0).values
def popular(uid):
    return to_recs(pop.astype(float), uid, "Popular")            # same bestseller list for everyone

In [ ]:
def item_cf(uid):
    user_vec = R[uid - 1].toarray().ravel()      # this user's training weights
    return to_recs(user_vec @ item_sim, uid, "Item-CF")

In [ ]:
from sentence_transformers import SentenceTransformer
encoder = SentenceTransformer("all-MiniLM-L6-v2")
texts = (products["category"] + ". " + products["product_name"] + ". " + products["description"]).tolist()
item_emb = encoder.encode(texts, normalize_embeddings=True)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
def content_emb(uid):
    mine = inter[inter["user_id"] == uid]                          # TRAINING interactions only
    w = mine["weight"].values[:, None]
    profile = (w * item_emb[mine["product_id"].values - 1]).sum(axis=0) / w.sum()
    profile = profile / np.linalg.norm(profile)
    return to_recs(item_emb @ profile, uid, "Content-Emb")

In [ ]:
def minmax(s):
    spread = s.max() - s.min()
    return (s - s.min()) / spread if spread > 0 else s * 0

def hybrid_v2(uid, alpha=0.5):
    cf = minmax(item_cf(uid).set_index("product_id")["score"])
    cb = minmax(content_emb(uid).set_index("product_id")["score"])
    combined = (alpha * cf + (1 - alpha) * cb).sort_values(ascending=False)
    return products.set_index("product_id").loc[combined.index].reset_index().assign(score=combined.values)

In [ ]:
import logging
from legacy.model import collaborative_filtering, content_based_filtering, hybrid_recommendation   # the ORIGINAL code
logging.getLogger().setLevel(logging.WARNING)          # hide model.py's debug messages

old_cf = lambda u: collaborative_filtering(u, train_purchases, products)
old_cb = lambda u: content_based_filtering(u, train_purchases, train_browsing, products).sort_values("score", ascending=False)
old_hy = lambda u: hybrid_recommendation(u, train_purchases, train_browsing, products)

In [ ]:
alphas = [0.0, 0.25, 0.5, 0.75, 1.0]
val_scores = {}
for a in alphas:
    val_scores[a] = evaluate(lambda u: hybrid_v2(u, alpha=a), val_users)["NDCG@10"]
    print(f"alpha = {a:.2f} -> validation NDCG@10 = {val_scores[a]:.3f}")

best_alpha = max(val_scores, key=val_scores.get)      # alpha with the highest validation NDCG
print("Best alpha:", best_alpha)

alpha = 0.00 -> validation NDCG@10 = 0.104
alpha = 0.25 -> validation NDCG@10 = 0.142
alpha = 0.50 -> validation NDCG@10 = 0.174
alpha = 0.75 -> validation NDCG@10 = 0.188
alpha = 1.00 -> validation NDCG@10 = 0.174
Best alpha: 0.75


In [ ]:
methods = {
    "Random": random_recs, "Popular": popular,
    "Old CF": old_cf, "Old Content": old_cb, "Old Hybrid": old_hy,
    "Item-CF": item_cf, "Content-Emb": content_emb,
    f"Hybrid (a={best_alpha})": lambda u: hybrid_v2(u, alpha=best_alpha),
}
rows = [{"Method": name, **evaluate(fn, test_users)} for name, fn in methods.items()]
results = pd.DataFrame(rows).set_index("Method").round(3)
results

,HR@10,NDCG@10
Method,,
Random,0.075,0.039
Popular,0.211,0.108
Old CF,0.197,0.109
Old Content,0.140,0.064
Old Hybrid,0.140,0.064
Item-CF,0.373,0.196
Content-Emb,0.232,0.103
Hybrid (a=0.75),0.364,0.197


In [ ]:
def avg_categories(fn, user_ids):
    """How many DIFFERENT categories appear in the top 10, on average (max possible = 8)."""
    return np.mean([fn(u).head(10)["category"].nunique() for u in user_ids])

for name in ["Popular", "Item-CF", "Content-Emb", f"Hybrid (a={best_alpha})"]:
    print(f"{name:22s}: {avg_categories(methods[name], test_users):.1f} categories in top-10")

Popular               : 6.3 categories in top-10
Item-CF               : 3.1 categories in top-10
Content-Emb           : 1.7 categories in top-10
Hybrid (a=0.75)       : 2.4 categories in top-10


## Day 3 results (real Myntra products) - test users only
Split: each user's last purchase hidden (users with >= 2 purchases); its browse
removed too (leakage). 455 users -> 227 validation (tune alpha) / 228 test.

| Method           | HR@10 | NDCG@10 |
|------------------|-------|---------|
| Random           | 0.075 | 0.039   |
| Popular          | 0.211 | 0.108   |
| Old CF           | 0.197 | 0.109   |
| Old Content      | 0.140 | 0.064   |
| Old Hybrid       | 0.140 | 0.064   |
| Item-CF          | 0.373 | 0.196   |
| Content-Emb      | 0.232 | 0.103   |
| Hybrid (a=0.75)  | 0.364 | 0.197   |  <- final model (chosen on validation)

Validation NDCG by alpha: 0->0.104, 0.25->0.142, 0.5->0.174, 0.75->0.188, 1.0->0.174
Diversity (avg categories in top-10): Popular 6.3 | Item-CF 3.1 | Hybrid 2.4 | Content 1.7

Takeaways:
- Final hybrid: +85% HR, +81% NDCG over original CF; ~1.7x Popular, ~5x Random
- Hybrid ties Item-CF on test (difference within ~+/-3 pt noise); kept hybrid because
  validation chose it (no test-set snooping) + content handles cold-start items
- Original CF ~ Popular; original content/hybrid worse than Popular
- Content adds accuracy on validation but lowers diversity (3.1 -> 2.4 categories)